# 02 Feature Extraction

## 1. Project setup

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

## 2. Imports

In [2]:
import pandas as pd

## 3. Paths and configuration

In [3]:
from src.config import FEATURES_CSV, METADATA_CSV, ensure_output_dirs, get_image_dir
from src.extract_features import load_or_build_features
from src.parse_metadata import build_metadata_table

ensure_output_dirs()
IMAGE_DIR = get_image_dir()
FORCE_REBUILD = False
SAMPLE_N = 50

## 4. Load metadata

In [4]:
if METADATA_CSV.exists():
    metadata = pd.read_csv(METADATA_CSV)
else:
    metadata = build_metadata_table(IMAGE_DIR)
    metadata.to_csv(METADATA_CSV, index=False)

metadata.head()

,filename,path,kv,mm,label,sample,area,mag,image_id,original_filename,area_group,param_group
0,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,/home/jp/MSE446_Nanoparticle_Ordering/data/fla...,10p0kV,11p3mm,ordered,S1,no_area,100k,100,100-S1-no_area-100k-ordered,S1__no_area,10p0kV__11p3mm__100k
1,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,/home/jp/MSE446_Nanoparticle_Ordering/data/fla...,10p0kV,11p3mm,ordered,S1,no_area,100k,101,101-S1-no_area-100k-ordered,S1__no_area,10p0kV__11p3mm__100k
2,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,/home/jp/MSE446_Nanoparticle_Ordering/data/fla...,10p0kV,11p3mm,ordered,S1,no_area,100k,102,102-S1-no_area-100k-ordered,S1__no_area,10p0kV__11p3mm__100k
3,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,/home/jp/MSE446_Nanoparticle_Ordering/data/fla...,10p0kV,11p3mm,ordered,S1,no_area,100k,103,103-S1-no_area-100k-ordered,S1__no_area,10p0kV__11p3mm__100k
4,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,/home/jp/MSE446_Nanoparticle_Ordering/data/fla...,10p0kV,11p3mm,ordered,S1,no_area,100k,104,104-S1-no_area-100k-ordered,S1__no_area,10p0kV__11p3mm__100k


## 5. Sanity checks

In [5]:
print(f"Metadata rows: {len(metadata)}")
print(f"Feature cache: {FEATURES_CSV}")
print(f"FORCE_REBUILD={FORCE_REBUILD}, SAMPLE_N={SAMPLE_N}")
assert metadata["label"].isin(["ordered", "disordered"]).all()

Metadata rows: 1000
Feature cache: /home/jp/MSE446_Nanoparticle_Ordering/data/features.csv
FORCE_REBUILD=False, SAMPLE_N=50


## 6. Main analysis

In [6]:
features = load_or_build_features(
    metadata,
    FEATURES_CSV,
    force_rebuild=FORCE_REBUILD,
    sample_n=SAMPLE_N,
)
features.head()

,filename,path,label,sample,area,group,kv,mm,mag,mean_intensity,...,hog_1754,hog_1755,hog_1756,hog_1757,hog_1758,hog_1759,hog_1760,hog_1761,hog_1762,hog_1763
0,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,/home/jp/MSE446_Nanoparticle_Ordering/flat_wit...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,0.354887,...,0.252466,0.136175,0.023808,0.035353,0.024533,0.092617,0.089103,0.242730,0.232808,0.121788
1,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,/home/jp/MSE446_Nanoparticle_Ordering/flat_wit...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,0.414919,...,0.238532,0.092513,0.044204,0.036822,0.041132,0.079723,0.112221,0.244662,0.244662,0.117387
2,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,/home/jp/MSE446_Nanoparticle_Ordering/flat_wit...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,0.311410,...,0.212062,0.162398,0.097374,0.051206,0.056435,0.217561,0.153203,0.138943,0.099091,0.103663
3,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,/home/jp/MSE446_Nanoparticle_Ordering/flat_wit...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,0.326260,...,0.237096,0.161114,0.032205,0.057718,0.045735,0.139168,0.178121,0.237096,0.237096,0.121838
4,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,/home/jp/MSE446_Nanoparticle_Ordering/flat_wit...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,0.260068,...,0.198071,0.108358,0.041262,0.059199,0.050300,0.244631,0.103342,0.216678,0.216456,0.145649


## 7. Results/figures

In [7]:
print(f"Feature table shape: {features.shape}")
print(features.filter(regex="^(mean|std|min|max|percentile|entropy|edge|bright)").describe().T)

Feature table shape: (50, 1789)
                    count      mean       std       min       25%       50%  \
mean_intensity       50.0  0.276634  0.071252  0.136557  0.222592  0.279214   
std_intensity        50.0  0.112765  0.025932  0.071198  0.094775  0.106509   
min_intensity        50.0  0.049771  0.052799  0.000000  0.000000  0.033333   
max_intensity        50.0  1.000000  0.000000  1.000000  1.000000  1.000000   
entropy              50.0  6.398265  0.438319  5.362995  6.123655  6.410994   
bright_pixel_ratio   50.0  0.098144  0.001101  0.096096  0.097360  0.098023   
percentile_1         50.0  0.121112  0.074174  0.000000  0.070588  0.117647   
percentile_5         50.0  0.152282  0.075741  0.011765  0.101961  0.152941   
percentile_10        50.0  0.170891  0.075404  0.027451  0.114706  0.174510   
percentile_25        50.0  0.205414  0.073445  0.058824  0.147059  0.201961   
percentile_50        50.0  0.250955  0.073896  0.109804  0.185294  0.241176   
percentile_75       

## 8. Notes for report

- Features are image-derived only.
- Metadata columns are retained only for labels, grouping, and audit checks.
- Set `SAMPLE_N = None` and `FORCE_REBUILD = True` when ready to compute all images.